# Laboratório de Ciência de Dados
## Aula 7 — Redução de dimensionalidade: PCA e visualização de dados em alta dimensão

Este notebook é **autoinstrucional**. Leia as explicações e execute as células na ordem.

### Situação

Na Aula 6, a VGAbank utilizou **K-Means** para segmentar clientes a partir de seis atributos.

O agrupamento foi realizado em um espaço de seis dimensões.

Mas surgiu um problema:

> **Como visualizar e interpretar dados quando existem muitas dimensões?**

Não conseguimos desenhar diretamente um gráfico com seis eixos espaciais.

Nesta aula utilizaremos **PCA — Principal Component Analysis (Análise de Componentes Principais)** para construir novas dimensões que concentrem grande parte da informação presente nos atributos originais.

### Objetivos de aprendizagem

Ao final da atividade, você deverá ser capaz de:

- explicar a ideia de redução de dimensionalidade;
- compreender a lógica das **componentes principais**;
- perceber por que a escala dos atributos influencia o PCA;
- implementar as etapas principais do PCA manualmente;
- aplicar `PCA` do `scikit-learn`;
- interpretar **variância explicada** e **variância acumulada**;
- decidir quantas componentes manter;
- interpretar os **loadings** das componentes;
- visualizar dados originalmente multidimensionais em 2D;
- relacionar PCA com o agrupamento estudado na Aula 6.

> PCA não “descobre as variáveis mais importantes”. Ele cria **novas variáveis**, chamadas componentes principais, como combinações dos atributos originais.

# 1. Preparando o ambiente

Utilizaremos `pandas`, `numpy`, `matplotlib` e ferramentas do `scikit-learn`.

As células de código estão comentadas para facilitar a interpretação.

In [ ]:
# Operações numéricas.
import numpy as np

# Manipulação de tabelas.
import pandas as pd

# Visualizações.
import matplotlib.pyplot as plt

# Padronização dos atributos.
from sklearn.preprocessing import StandardScaler

# Implementação do PCA.
from sklearn.decomposition import PCA

# K-Means será usado apenas para conectar esta aula à Aula 6.
from sklearn.cluster import KMeans

# Mantém os resultados reproduzíveis.
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 2. Retomando os dados da Aula 6

Usaremos novamente uma base simulada de clientes da VGAbank.

Os atributos são:

- `renda_mensal`;
- `gasto_cartao`;
- `transacoes_mes`;
- `recencia_dias`;
- `saldo_medio`;
- `uso_app_mes`.

Temos, portanto, **seis dimensões**.

In [ ]:
def gerar_clientes(n, medias, desvios):
    # Gera n observações com as médias e desvios definidos.
    return rng.normal(
        loc=np.array(medias),
        scale=np.array(desvios),
        size=(n, len(medias))
    )


# Quatro perfis comportamentais usados apenas para gerar a base.
segmento_1 = gerar_clientes(
    230,
    medias=[4500, 1200, 9, 38, 3500, 6],
    desvios=[900, 450, 3, 12, 1400, 3]
)

segmento_2 = gerar_clientes(
    240,
    medias=[8500, 3600, 23, 12, 13000, 18],
    desvios=[1400, 900, 5, 5, 3500, 5]
)

segmento_3 = gerar_clientes(
    210,
    medias=[16000, 7200, 28, 6, 42000, 24],
    desvios=[2600, 1500, 6, 3, 8000, 6]
)

segmento_4 = gerar_clientes(
    220,
    medias=[6000, 3000, 38, 4, 6500, 48],
    desvios=[1100, 750, 7, 2, 1800, 8]
)

# Junta os quatro blocos.
dados_clientes = np.vstack([
    segmento_1,
    segmento_2,
    segmento_3,
    segmento_4
])

colunas = [
    "renda_mensal",
    "gasto_cartao",
    "transacoes_mes",
    "recencia_dias",
    "saldo_medio",
    "uso_app_mes"
]

df = pd.DataFrame(
    dados_clientes,
    columns=colunas
)

# Impede valores negativos sem sentido no contexto.
for coluna in colunas:
    df[coluna] = df[coluna].clip(lower=0)

# Arredondamentos apenas para facilitar a leitura.
df["renda_mensal"] = df["renda_mensal"].round(2)
df["gasto_cartao"] = df["gasto_cartao"].round(2)
df["transacoes_mes"] = df["transacoes_mes"].round(0)
df["recencia_dias"] = df["recencia_dias"].round(0)
df["saldo_medio"] = df["saldo_medio"].round(2)
df["uso_app_mes"] = df["uso_app_mes"].round(0)

df.head()

In [ ]:
print("Número de clientes:", len(df))
print("Número de atributos:", len(colunas))

df.describe().round(2)

### Pergunta 1

Se tentássemos representar cada atributo como um eixo espacial, quantas dimensões seriam necessárias?

Por que um gráfico de dispersão comum não consegue mostrar diretamente toda a estrutura desse conjunto?

# 3. O problema da alta dimensionalidade

Com dois atributos, podemos construir um gráfico 2D.

Com três, ainda podemos utilizar um gráfico 3D.

Com seis, vinte ou centenas de atributos, a visualização direta deixa de ser possível.

Além disso, atributos diferentes podem carregar informações parcialmente redundantes.

Por exemplo:

- clientes com renda maior podem apresentar saldo médio maior;
- maior atividade pode aparecer simultaneamente em `transacoes_mes` e `uso_app_mes`.

A redução de dimensionalidade procura representar os dados com menos dimensões, tentando preservar informação relevante.

In [ ]:
# Calcula a correlação linear entre os atributos.
correlacoes = df[colunas].corr()

correlacoes.round(2)

In [ ]:
# Visualização simples da matriz de correlação.
fig, ax = plt.subplots(figsize=(7, 6))

imagem = ax.imshow(
    correlacoes.values,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    range(len(colunas)),
    labels=colunas,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(len(colunas)),
    labels=colunas
)

# Escreve o valor de cada correlação dentro da matriz.
for i in range(len(colunas)):
    for j in range(len(colunas)):
        ax.text(
            j,
            i,
            f"{correlacoes.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(imagem, ax=ax, label="Correlação")
plt.title("Correlação entre os atributos")
plt.tight_layout()
plt.show()

### Pergunta 2

Quais pares de atributos apresentam correlação mais intensa?

Se dois atributos contêm informação semelhante, seria possível representar parte dessa informação usando uma única nova dimensão?

# 4. Ideia central do PCA

O PCA cria novos eixos chamados **componentes principais**.

A primeira componente, **PC1**, procura a direção em que os dados apresentam a maior variância possível.

A segunda componente, **PC2**, procura a maior variância restante, com a condição de ser ortogonal à primeira.

O processo continua para as componentes seguintes.

Assim:

```text
atributos originais
        ↓
combinações lineares
        ↓
PC1, PC2, PC3, ...
```

As componentes são ordenadas:

$$
\text{variância}(PC1)
\geq
\text{variância}(PC2)
\geq
\text{variância}(PC3)
\geq \cdots
$$

Se as primeiras componentes concentrarem grande parte da variabilidade dos dados, podemos utilizar menos dimensões.

## 4.1 Um exemplo geométrico simples

Vamos observar um conjunto de dados bidimensional altamente correlacionado.

Mesmo possuindo dois atributos, grande parte de sua variação ocorre em uma única direção.

In [ ]:
# Cria um primeiro atributo.
x1 = rng.normal(0, 2, 200)

# O segundo atributo é fortemente relacionado ao primeiro.
x2 = 1.8 * x1 + rng.normal(0, 0.8, 200)

X_exemplo = np.column_stack([
    x1,
    x2
])

plt.figure(figsize=(6, 5))

plt.scatter(
    X_exemplo[:, 0],
    X_exemplo[:, 1],
    alpha=0.6
)

plt.xlabel("Atributo 1")
plt.ylabel("Atributo 2")
plt.title("Duas variáveis fortemente relacionadas")
plt.grid(alpha=0.2)
plt.show()

Observe que os pontos formam aproximadamente uma faixa diagonal.

Uma nova dimensão orientada ao longo dessa faixa conseguiria representar boa parte da variação observada.

Essa é a intuição geométrica do PCA.

# 5. Antes do PCA: a escala importa

Na Aula 6 vimos que métodos baseados em distâncias podem ser influenciados pela escala dos atributos.

O PCA também é sensível à escala.

Observe:

- `saldo_medio` pode assumir dezenas de milhares;
- `recencia_dias` assume valores muito menores.

Se aplicarmos PCA diretamente, atributos de maior variância numérica podem dominar as componentes.

Por isso, quando as unidades são diferentes, é comum aplicar **padronização Z-score** antes do PCA.

In [ ]:
# Cria o padronizador.
scaler = StandardScaler()

# Aprende média e desvio padrão de cada atributo
# e transforma os dados.
X_std = scaler.fit_transform(
    df[colunas]
)

# Converte para DataFrame apenas para inspeção.
df_std = pd.DataFrame(
    X_std,
    columns=colunas
)

df_std.agg([
    "mean",
    "std"
]).round(3)

Após a padronização:

$$
z =
\frac{x-\mu}{\sigma}
$$

cada atributo passa a ser interpretado aproximadamente em unidades de desvio padrão.

Isso evita que a unidade original determine sozinha a importância no PCA.

# 6. Implementando as etapas principais do PCA

Antes de utilizar `sklearn`, vamos construir as etapas centrais manualmente.

Como os dados já estão padronizados, eles estão centrados aproximadamente em zero.

O processo será:

1. calcular a matriz de covariância;
2. obter autovalores e autovetores;
3. ordenar as direções pela variância;
4. projetar os dados nas componentes principais.

## 6.1 Matriz de covariância

A matriz de covariância descreve como as variáveis variam conjuntamente.

Como os atributos estão padronizados, ela será muito próxima da matriz de correlação.

In [ ]:
# rowvar=False informa que:
# linhas = observações
# colunas = atributos
matriz_cov = np.cov(
    X_std,
    rowvar=False
)

df_cov = pd.DataFrame(
    matriz_cov,
    index=colunas,
    columns=colunas
)

df_cov.round(2)

## 6.2 Autovalores e autovetores

Os **autovetores** indicam as direções das componentes principais.

Os **autovalores** indicam quanta variância existe em cada uma dessas direções.

Como a matriz de covariância é simétrica, podemos usar `np.linalg.eigh()`.

In [ ]:
# Calcula autovalores e autovetores.
autovalores, autovetores = np.linalg.eigh(
    matriz_cov
)

print("Autovalores antes da ordenação:")
print(
    np.round(
        autovalores,
        3
    )
)

Os valores retornados não estão necessariamente em ordem decrescente.

Como queremos começar pela direção de maior variância, precisamos ordenar.

In [ ]:
# argsort() retorna os índices que ordenariam os valores.
# [::-1] inverte a ordem para decrescente.
ordem = np.argsort(
    autovalores
)[::-1]

# Reordena os autovalores.
autovalores_ord = autovalores[
    ordem
]

# Reordena também os autovetores correspondentes.
autovetores_ord = autovetores[
    :,
    ordem
]

print("Autovalores ordenados:")
print(
    np.round(
        autovalores_ord,
        3
    )
)

# 7. Variância explicada

Cada autovalor representa a quantidade de variância associada a uma componente.

A proporção explicada pela componente \(i\) é:

$$
\text{Variância explicada}_i =
\frac{\lambda_i}
{\sum_{j=1}^{p}\lambda_j}
$$

onde \(\lambda_i\) é o autovalor da componente.

In [ ]:
# Divide cada autovalor pela soma total.
variancia_explicada_manual = (
    autovalores_ord
    / autovalores_ord.sum()
)

# Calcula também a variância acumulada.
variancia_acumulada_manual = np.cumsum(
    variancia_explicada_manual
)

df_variancia_manual = pd.DataFrame({
    "componente": [
        f"PC{i}"
        for i in range(
            1,
            len(colunas) + 1
        )
    ],
    "variancia_explicada": variancia_explicada_manual,
    "variancia_acumulada": variancia_acumulada_manual
})

df_variancia_manual.round(4)

### Pergunta 3

Quanto da variância é explicada por:

- PC1?
- PC1 + PC2?
- PC1 + PC2 + PC3?

Seria possível reduzir seis atributos para duas ou três componentes sem perder toda a estrutura dos dados?

# 8. Projetando os dados nas componentes

Para representar os dados nas duas primeiras componentes, utilizaremos os dois primeiros autovetores.

A multiplicação abaixo transforma cada observação:

```text
6 atributos originais → PC1 e PC2
```

In [ ]:
# Seleciona os autovetores das duas primeiras componentes.
W_2 = autovetores_ord[
    :,
    :2
]

# Projeta as observações padronizadas
# no novo espaço de duas dimensões.
X_pca_manual = X_std @ W_2

df_pca_manual = pd.DataFrame(
    X_pca_manual,
    columns=[
        "PC1",
        "PC2"
    ]
)

df_pca_manual.head()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_pca_manual["PC1"],
    df_pca_manual["PC2"],
    alpha=0.6
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Projeção manual nas duas primeiras componentes")
plt.grid(alpha=0.2)
plt.show()

Agora conseguimos representar em um gráfico 2D observações que originalmente possuíam **seis atributos**.

Isso envolve perda de informação, porque descartamos as componentes restantes.

A pergunta importante é:

> **quanto de informação foi preservado?**

A variância explicada nos ajuda a responder.

# 9. PCA com `scikit-learn`

Em aplicações reais, normalmente utilizamos uma implementação consolidada.

Vamos repetir o procedimento com `sklearn.decomposition.PCA`.

In [ ]:
# Mantém inicialmente todas as componentes.
pca_completo = PCA()

# Ajusta o PCA e transforma os dados.
X_pca_completo = pca_completo.fit_transform(
    X_std
)

# Proporção da variância explicada por cada componente.
pca_completo.explained_variance_ratio_

In [ ]:
df_variancia = pd.DataFrame({
    "componente": [
        f"PC{i}"
        for i in range(
            1,
            len(colunas) + 1
        )
    ],
    "variancia_explicada": pca_completo.explained_variance_ratio_,
    "variancia_acumulada": np.cumsum(
        pca_completo.explained_variance_ratio_
    )
})

# Mostra em porcentagem para facilitar a interpretação.
df_variancia["variancia_explicada_pct"] = (
    df_variancia["variancia_explicada"]
    * 100
)

df_variancia["variancia_acumulada_pct"] = (
    df_variancia["variancia_acumulada"]
    * 100
)

df_variancia.round(2)

Os resultados devem ser equivalentes aos obtidos manualmente.

O sinal de uma componente pode aparecer invertido em relação à implementação manual. Isso não representa erro: um autovetor e seu negativo descrevem o mesmo eixo.

# 10. Visualizando a variância explicada

Um gráfico comum é o **scree plot**, que mostra quanto cada componente explica.

Também é útil observar a variância acumulada.

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    df_variancia["componente"],
    df_variancia["variancia_explicada_pct"]
)

plt.xlabel("Componente principal")
plt.ylabel("Variância explicada (%)")
plt.title("Variância explicada por componente")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    df_variancia["componente"],
    df_variancia["variancia_acumulada_pct"],
    marker="o"
)

plt.axhline(
    90,
    linestyle="--",
    label="90%"
)

plt.axhline(
    95,
    linestyle="--",
    label="95%"
)

plt.xlabel("Número de componentes")
plt.ylabel("Variância acumulada (%)")
plt.title("Variância explicada acumulada")
plt.ylim(0, 105)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Pergunta 4

Quantas componentes são necessárias para explicar aproximadamente:

- 80% da variância?
- 90%?
- 95%?

Não existe um percentual universalmente obrigatório. A escolha depende do objetivo da análise.

# 11. Escolhendo automaticamente pela variância

O `PCA` pode receber um valor entre 0 e 1 em `n_components`.

Por exemplo:

```python
PCA(n_components=0.90)
```

significa:

> mantenha o menor número de componentes necessário para explicar pelo menos 90% da variância.

In [ ]:
pca_90 = PCA(
    n_components=0.90
)

X_pca_90 = pca_90.fit_transform(
    X_std
)

print(
    "Número de componentes mantidas:",
    pca_90.n_components_
)

print(
    "Variância total preservada:",
    round(
        pca_90.explained_variance_ratio_.sum()
        * 100,
        2
    ),
    "%"
)

print(
    "Formato original:",
    X_std.shape
)

print(
    "Formato reduzido:",
    X_pca_90.shape
)

Essa é a ideia de **redução de dimensionalidade**:

o número de colunas diminui, enquanto tentamos preservar uma proporção elevada da variância original.

# 12. Visualização em duas dimensões

Agora construiremos explicitamente um PCA com duas componentes.

Essa escolha é útil para visualização.

Não estamos afirmando que duas componentes sejam necessariamente suficientes para qualquer etapa de modelagem.

In [ ]:
pca_2 = PCA(
    n_components=2
)

X_pca_2 = pca_2.fit_transform(
    X_std
)

df_pca_2 = pd.DataFrame(
    X_pca_2,
    columns=[
        "PC1",
        "PC2"
    ]
)

print(
    "Variância preservada em 2D:",
    round(
        pca_2.explained_variance_ratio_.sum()
        * 100,
        2
    ),
    "%"
)

df_pca_2.head()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_pca_2["PC1"],
    df_pca_2["PC2"],
    alpha=0.6
)

plt.xlabel(
    f"PC1 ({pca_2.explained_variance_ratio_[0] * 100:.1f}%)"
)

plt.ylabel(
    f"PC2 ({pca_2.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title(
    "Clientes da VGAbank projetados em duas dimensões"
)

plt.grid(alpha=0.2)
plt.show()

### Pergunta 5

Você consegue perceber regiões ou estruturas no gráfico?

Lembre-se:

> PCA não é um algoritmo de clustering.

Ele apenas criou uma representação em dimensões que concentram variância.

# 13. O que significa cada componente?

PC1 e PC2 não correspondem diretamente a nenhuma variável original.

Cada componente é uma **combinação linear** dos atributos.

Os coeficientes associados aos atributos são frequentemente chamados de **loadings**.

Vamos inspecioná-los.

In [ ]:
# components_ possui:
# linhas = componentes
# colunas = atributos originais
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=colunas,
    columns=[
        "PC1",
        "PC2"
    ]
)

loadings.round(3)

## Como interpretar os loadings?

Valores com maior magnitude indicam atributos que contribuem mais para aquela direção.

O sinal indica a direção da relação dentro da componente.

Por exemplo:

- dois atributos com sinais semelhantes tendem a contribuir na mesma direção;
- sinais opostos indicam contribuições em sentidos opostos.

A interpretação deve considerar **magnitude, sinal e contexto**.

In [ ]:
# Visualiza os loadings das duas primeiras componentes.
loadings.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel("Loading")
plt.xlabel("Atributo")
plt.title("Contribuição dos atributos para PC1 e PC2")
plt.xticks(
    rotation=45,
    ha="right"
)
plt.tight_layout()
plt.show()

### Pergunta 6

Quais atributos apresentam maior magnitude em PC1?

E em PC2?

Com base nisso, como você descreveria informalmente cada componente?

# 14. Conectando com a Aula 6: PCA + K-Means

Na aula anterior, o K-Means encontrou grupos utilizando os seis atributos padronizados.

Vamos repetir o K-Means com `k=4` e utilizar os rótulos **apenas para colorir** a projeção PCA.

O clustering continuará sendo calculado no espaço original de seis dimensões.

In [ ]:
# Executa K-Means usando os seis atributos padronizados.
kmeans = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)

clusters = kmeans.fit_predict(
    X_std
)

# Adiciona os rótulos apenas para visualização.
df_pca_2["cluster"] = clusters

df_pca_2.head()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df_pca_2["PC1"],
    df_pca_2["PC2"],
    c=df_pca_2["cluster"],
    alpha=0.65
)

plt.xlabel(
    f"PC1 ({pca_2.explained_variance_ratio_[0] * 100:.1f}%)"
)

plt.ylabel(
    f"PC2 ({pca_2.explained_variance_ratio_[1] * 100:.1f}%)"
)

plt.title(
    "Visualização dos clusters da Aula 6 no espaço PCA"
)

plt.grid(alpha=0.2)
plt.show()

### Pergunta 7

Agora os grupos parecem mais evidentes?

Se dois clusters apresentarem sobreposição na projeção 2D, isso prova que são iguais?

Não.

Parte da informação permanece em componentes que não aparecem no gráfico.

## 14.1 Projetando os centroides

Também podemos transformar os centroides do K-Means para o mesmo espaço PCA.

Como o K-Means foi ajustado nos dados padronizados, seus centroides podem ser transformados diretamente pelo PCA.

In [ ]:
# Projeta os centroides do espaço de seis dimensões
# para o plano PC1 × PC2.
centroides_pca = pca_2.transform(
    kmeans.cluster_centers_
)

plt.figure(figsize=(7, 5))

plt.scatter(
    df_pca_2["PC1"],
    df_pca_2["PC2"],
    c=df_pca_2["cluster"],
    alpha=0.55
)

plt.scatter(
    centroides_pca[:, 0],
    centroides_pca[:, 1],
    marker="X",
    s=220
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(
    "Clusters e centroides projetados no espaço PCA"
)

plt.grid(alpha=0.2)
plt.show()

# 15. PCA pode ser usado antes do algoritmo?

Sim.

Em bases com muitas variáveis, PCA pode ser utilizado antes de outros algoritmos para:

- reduzir dimensionalidade;
- diminuir redundância;
- reduzir custo computacional;
- atenuar parte do ruído;
- contornar alguns problemas associados a espaços de dimensão muito alta.

Mas existe uma troca:

> ao remover componentes, também removemos informação.

Vamos comparar o número original de dimensões com uma versão que preserva pelo menos 90% da variância.

In [ ]:
print(
    "Dimensões originais:",
    X_std.shape[1]
)

print(
    "Dimensões após PCA 90%:",
    X_pca_90.shape[1]
)

print(
    "Redução:",
    X_std.shape[1] - X_pca_90.shape[1],
    "dimensão(ões)"
)

Nesta base existem apenas seis atributos, portanto o ganho computacional é pequeno.

Em uma base com centenas ou milhares de atributos, a redução pode ser muito mais significativa.

# 16. Redução significa perda de informação

Uma forma de visualizar a perda é reconstruir aproximadamente os dados originais a partir das componentes mantidas.

Quanto menos componentes utilizarmos, maior tende a ser o erro de reconstrução.

In [ ]:
erros_reconstrucao = []

for n_componentes in range(
    1,
    len(colunas) + 1
):
    # Ajusta PCA com determinado número de componentes.
    pca_temp = PCA(
        n_components=n_componentes
    )

    # Reduz os dados.
    reduzido = pca_temp.fit_transform(
        X_std
    )

    # Tenta reconstruir os dados padronizados originais.
    reconstruido = pca_temp.inverse_transform(
        reduzido
    )

    # Erro quadrático médio de reconstrução.
    mse = np.mean(
        (X_std - reconstruido) ** 2
    )

    erros_reconstrucao.append({
        "componentes": n_componentes,
        "erro_reconstrucao": mse
    })

df_erros = pd.DataFrame(
    erros_reconstrucao
)

df_erros.round(4)

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    df_erros["componentes"],
    df_erros["erro_reconstrucao"],
    marker="o"
)

plt.xlabel("Número de componentes mantidas")
plt.ylabel("Erro médio de reconstrução")
plt.title("Dimensionalidade × perda de informação")
plt.xticks(df_erros["componentes"])
plt.grid(alpha=0.3)
plt.show()

### Pergunta 8

O que acontece com o erro quando aumentamos o número de componentes?

Quando mantemos todas as componentes, por que o erro se aproxima de zero?

# 17. Visualização em três dimensões

Duas componentes são convenientes para gráficos.

Podemos também utilizar três componentes para uma visualização 3D.

Essa visualização preserva mais variância, mas geralmente é menos simples de interpretar em materiais estáticos.

In [ ]:
# PCA com três componentes.
pca_3 = PCA(
    n_components=3
)

X_pca_3 = pca_3.fit_transform(
    X_std
)

print(
    "Variância preservada em 3D:",
    round(
        pca_3.explained_variance_ratio_.sum()
        * 100,
        2
    ),
    "%"
)

In [ ]:
# Cria um eixo tridimensional.
fig = plt.figure(
    figsize=(8, 6)
)

ax = fig.add_subplot(
    111,
    projection="3d"
)

ax.scatter(
    X_pca_3[:, 0],
    X_pca_3[:, 1],
    X_pca_3[:, 2],
    c=clusters,
    alpha=0.6
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title(
    "Visualização PCA em três dimensões"
)

plt.show()

# 18. Limitações e cuidados com PCA

PCA é útil, mas não deve ser tratado como uma transformação universal.

**1. PCA é linear.**  
Estruturas não lineares podem não ser bem representadas.

**2. As componentes podem ser difíceis de interpretar.**  
Elas combinam vários atributos.

**3. Escala importa.**  
Quando os atributos usam unidades muito diferentes, a padronização costuma ser necessária.

**4. Variância não é sinônimo de relevância.**  
Uma dimensão de baixa variância ainda pode ser importante para uma tarefa específica.

**5. Redução implica perda.**  
Ao descartar componentes, descartamos parte da informação.

**6. PCA não substitui conhecimento do domínio.**  
Uma boa representação matemática não garante que a interpretação seja cientificamente ou operacionalmente adequada.

# 19. Exercícios

## Exercício 1 — PCA sem padronização

Aplique `PCA()` diretamente aos atributos originais, **sem utilizar `StandardScaler`**.

Depois responda:

1. qual componente concentra a maior variância?
2. quais atributos dominam PC1?
3. por que o resultado difere do PCA padronizado?

In [ ]:
# Complete aqui.

## Exercício 2 — escolha do número de componentes

Utilize:

```python
PCA(n_components=0.95)
```

Descubra:

- quantas componentes foram mantidas;
- qual a variância acumulada obtida;
- quantas dimensões foram eliminadas.

In [ ]:
# Complete aqui.

## Exercício 3 — interpretando componentes

Utilize os loadings do PCA completo.

Para **PC1, PC2 e PC3**:

1. identifique os três atributos com maior loading em módulo;
2. observe seus sinais;
3. proponha uma interpretação resumida para cada componente.

In [ ]:
# Complete aqui.

## Exercício 4 — visualização dos clusters

Construa novamente o gráfico `PC1 × PC2`, mas:

- utilize `k=3` no K-Means;
- depois utilize `k=5`.

Compare visualmente com o resultado de `k=4`.

O PCA determina qual valor de \(k\) é correto? Explique.

In [ ]:
# Complete aqui.

# 20. Desafio final — relatório de redução de dimensionalidade

Imagine que você precisa explicar os resultados para a equipe de dados da VGAbank.

Preencha:

**Número de atributos originais:** ____________________________

**Variância explicada por PC1:** ______________________________

**Variância acumulada por PC1 + PC2:** ________________________

**Número de componentes necessário para pelo menos 90%:** _____

**Atributos com maior contribuição para PC1:**  
_____________________________________________________________

**Interpretação proposta para PC1:**  
_____________________________________________________________

**Interpretação proposta para PC2:**  
_____________________________________________________________

**A projeção em 2D representa 100% da informação?**  
_____________________________________________________________

**Em que situação você utilizaria PCA antes de outro algoritmo?**  
_____________________________________________________________

**Qual cuidado você tomaria antes de aplicar PCA a atributos em escalas muito diferentes?**  
_____________________________________________________________

# 21. Reflexão final

Responda brevemente:

**1. O que significa reduzir dimensionalidade?**

**2. O que representa a primeira componente principal?**

**3. Por que PC1 e PC2 são diferentes dos atributos originais?**

**4. O que é variância explicada?**

**5. Por que normalmente padronizamos atributos antes do PCA quando suas escalas são diferentes?**

**6. O que os loadings ajudam a interpretar?**

**7. PCA é um algoritmo de clustering?**

**8. Uma projeção 2D contendo 80% da variância contém todas as informações da base?**

**9. Por que uma componente de baixa variância ainda poderia ser relevante para alguma tarefa?**

**10. Qual é a principal vantagem de usar PCA para visualizar dados de alta dimensão?**

# 22. Conclusão

Nesta aula percorremos:

**dados multidimensionais → padronização → matriz de covariância → autovalores/autovetores → componentes principais → variância explicada → redução → loadings → visualização**

Também conectamos o PCA ao K-Means estudado na Aula 6.

A ideia principal é:

> **PCA transforma vários atributos correlacionados em novas dimensões ordenadas pela quantidade de variância que conseguem representar.**

Isso permite reduzir e visualizar dados complexos, mas exige uma decisão sobre quanta informação podemos aceitar perder.

Em Ciência de Dados, reduzir dimensionalidade não é simplesmente “eliminar colunas”.

É construir uma representação mais compacta preservando, tanto quanto possível, a estrutura relevante dos dados.